# Neural Network for Oil Recovery Prediction
**Training dataset:** `proxy4.csv` | **External validation:** `External.csv`  
**Workflow:** EDA → Preprocessing → K-Fold CV → Full retrain → External validation

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

# ── plotting style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'CPU only')

## 1. Load & Inspect Data

In [ ]:
# ── Update filenames here if needed ──────────────────────────────────────────
PROXY_FILE    = 'proxy4.csv'    # change to e.g. 'Proxy5.csv' if needed
EXTERNAL_FILE = 'External.csv'

def read_csv_safe(path):
    """Try UTF-8 first, fall back to latin-1 (handles Excel/Windows-encoded CSVs)."""
    try:
        return pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        print(f"  UTF-8 failed for {path} — retrying with latin-1")
        return pd.read_csv(path, encoding='latin-1')

proxy4 = read_csv_safe(PROXY_FILE)
print(f'proxy4  shape : {proxy4.shape}')
print(f'Columns       : {list(proxy4.columns)}')
proxy4.head()

In [ ]:
proxy4.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
print('Missing values per column:')
print(proxy4.isnull().sum())

## 2. Identify Target Column

In [ ]:
# ── Auto-detect the oil recovery target column ────────────────────────────────
# Priority: column whose name contains 'recov' (case-insensitive)
# Fallback : last column
_recovery_candidates = [
    c for c in proxy4.columns
    if any(kw in c.lower() for kw in ('recov', 'rf', 'orf', 'oil_rec', 'recovery_factor'))
]

TARGET_COL = _recovery_candidates[0] if _recovery_candidates else proxy4.columns[-1]
FEATURE_COLS = [c for c in proxy4.columns if c != TARGET_COL]

print(f'Target   : {TARGET_COL}')
print(f'Features : {FEATURE_COLS}')
print(f'Samples  : {len(proxy4)}')

## 3. Exploratory Data Analysis

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(8, len(proxy4.columns)//2), max(6, len(proxy4.columns)//2)))
corr = proxy4.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature distributions vs target ──────────────────────────────────────────
n_feat = len(FEATURE_COLS)
ncols  = min(4, n_feat)
nrows  = int(np.ceil(n_feat / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.array(axes).ravel()

for i, col in enumerate(FEATURE_COLS):
    axes[i].scatter(proxy4[col], proxy4[TARGET_COL],
                    alpha=0.5, s=18, c=proxy4[TARGET_COL], cmap='viridis')
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel(TARGET_COL, fontsize=9)
    axes[i].set_title(f'{col} vs Recovery', fontsize=9, fontweight='bold')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature vs Oil Recovery Scatter Plots', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('eda_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(proxy4[TARGET_COL], bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.axvline(proxy4[TARGET_COL].mean(), color='red', lw=2, label=f'Mean={proxy4[TARGET_COL].mean():.3f}')
ax1.set_xlabel(TARGET_COL)
ax1.set_title('Target Distribution (proxy4)', fontweight='bold')
ax1.legend()

ax2.boxplot(proxy4[TARGET_COL], vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
ax2.set_ylabel(TARGET_COL)
ax2.set_title('Target Boxplot', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target_dist.png', bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# ── Drop rows with NaN, reset index ──────────────────────────────────────────
data = proxy4[FEATURE_COLS + [TARGET_COL]].dropna().reset_index(drop=True)
print(f'Rows after dropping NaN: {len(data)}')

X_all = data[FEATURE_COLS].values.astype(np.float32)
y_all = data[TARGET_COL].values.astype(np.float32).reshape(-1, 1)

print(f'X shape: {X_all.shape}')
print(f'y shape: {y_all.shape}')
print(f'y range: [{y_all.min():.4f}, {y_all.max():.4f}]')

## 5. Neural Network Architecture

In [ ]:
def build_nn(n_inputs: int,
             hidden_units: list = [128, 128, 64, 32],
             dropout_rate: float = 0.15,
             lr: float = 1e-3) -> keras.Model:
    """
    Feedforward NN for regression.
    Architecture: Dense + BatchNorm + ReLU + Dropout (repeated) → Linear output.
    """
    inp = keras.Input(shape=(n_inputs,), name='features')
    x   = inp

    for units in hidden_units:
        x = keras.layers.Dense(
            units,
            kernel_initializer='he_normal',
            kernel_regularizer=keras.regularizers.l2(1e-4)
        )(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation('relu')(x)
        x = keras.layers.Dropout(dropout_rate)(x)

    out = keras.layers.Dense(1, name='recovery')(x)

    model = keras.Model(inp, out, name='NN_OilRecovery')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',
        metrics=[keras.metrics.RootMeanSquaredError(name='rmse'),
                 keras.metrics.MeanAbsoluteError(name='mae')]
    )
    return model


# ── Print summary for reference ───────────────────────────────────────────────
demo_model = build_nn(X_all.shape[1])
demo_model.summary()
del demo_model


# ── Training helpers ──────────────────────────────────────────────────────────
EPOCHS     = 500
BATCH_SIZE = min(64, len(X_all) // 5)
PATIENCE   = 60

def make_callbacks(monitor='val_loss', restore=True):
    return [
        keras.callbacks.EarlyStopping(
            monitor=monitor, patience=PATIENCE,
            restore_best_weights=restore, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5,
            patience=PATIENCE // 3, min_lr=1e-6, verbose=0
        )
    ]

## 6. K-Fold Cross-Validation

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = y_true.ravel()
    y_pred = y_pred.ravel()
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {'R2': r2, 'RMSE': rmse, 'MAPE': mape}


N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_metrics   = []
fold_histories = []
fold_scalers   = []       # store scaler per fold for reference
oof_preds      = np.zeros_like(y_all)   # out-of-fold predictions

print(f'Running {N_FOLDS}-fold cross-validation  ({len(X_all)} samples)\n')
print(f'{"Fold":>5}  {"R²":>8}  {"RMSE":>10}  {"MAPE %":>10}  {"Best epoch":>12}')
print('-' * 55)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_all), 1):
    X_tr, X_va = X_all[train_idx], X_all[val_idx]
    y_tr, y_va = y_all[train_idx], y_all[val_idx]

    # ── Scale per fold (fit on train only) ────────────────────────────────
    sx, sy = StandardScaler(), StandardScaler()
    X_tr_s = sx.fit_transform(X_tr)
    X_va_s = sx.transform(X_va)
    y_tr_s = sy.fit_transform(y_tr)
    y_va_s = sy.transform(y_va)

    fold_scalers.append((sx, sy))

    # ── Build & train ──────────────────────────────────────────────────────
    tf.random.set_seed(42 + fold)
    model = build_nn(X_tr_s.shape[1])
    hist  = model.fit(
        X_tr_s, y_tr_s,
        validation_data=(X_va_s, y_va_s),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(),
        verbose=0
    )

    # ── Predict & inverse-scale ────────────────────────────────────────────
    y_pred_s = model.predict(X_va_s, verbose=0)
    y_pred   = sy.inverse_transform(y_pred_s)

    oof_preds[val_idx] = y_pred

    m = compute_metrics(y_va, y_pred)
    fold_metrics.append(m)
    fold_histories.append(hist.history)

    best_ep = np.argmin(hist.history['val_loss']) + 1
    print(f'{fold:>5}  {m["R2"]:>8.4f}  {m["RMSE"]:>10.5f}  {m["MAPE"]:>10.2f}  {best_ep:>12}')

    tf.keras.backend.clear_session()

# ── Summary ────────────────────────────────────────────────────────────────
print('-' * 55)
for metric in ('R2', 'RMSE', 'MAPE'):
    vals = [m[metric] for m in fold_metrics]
    print(f'{metric:>8}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  '
          f'[{np.min(vals):.4f}, {np.max(vals):.4f}]')

## 7. Cross-Validation Results

In [ ]:
# ── Metrics summary table ─────────────────────────────────────────────────────
cv_df = pd.DataFrame(fold_metrics, index=[f'Fold {i}' for i in range(1, N_FOLDS+1)])
cv_df.loc['Mean'] = cv_df.mean()
cv_df.loc['Std']  = cv_df.std()
cv_df.style.format('{:.4f}').background_gradient(cmap='RdYlGn', subset=['R2']).background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAPE'])

In [ ]:
# ── Training curves per fold ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, N_FOLDS, figsize=(5*N_FOLDS, 4), sharey=False)

for i, (hist, ax) in enumerate(zip(fold_histories, axes), 1):
    ax.semilogy(hist['loss'],     color='steelblue', lw=1.5, label='Train')
    ax.semilogy(hist['val_loss'], color='tomato',    lw=1.5, label='Val', alpha=0.85)
    best = np.argmin(hist['val_loss'])
    ax.axvline(best, color='gray', lw=1, ls='--')
    ax.set_title(f'Fold {i}  (best ep {best+1})', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8)

fig.suptitle(f'{N_FOLDS}-Fold CV — Training Curves', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('cv_training_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Out-of-fold parity plot ───────────────────────────────────────────────────
oof_m = compute_metrics(y_all, oof_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(y_all.ravel(), oof_preds.ravel(),
                c=np.arange(len(y_all)), cmap='viridis',
                alpha=0.7, s=30, edgecolors='none')
lo = min(y_all.min(), oof_preds.min())
hi = max(y_all.max(), oof_preds.max())
pad = (hi - lo) * 0.05
ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], 'k--', lw=2, label='Ideal 1:1')
ax.set_xlim([lo-pad, hi+pad])
ax.set_ylim([lo-pad, hi+pad])
ax.set_xlabel(f'Actual {TARGET_COL}', fontsize=11)
ax.set_ylabel(f'Predicted {TARGET_COL}', fontsize=11)
ax.set_title(
    f'Out-of-Fold Parity Plot  ({N_FOLDS}-Fold CV)\n'
    f'R²={oof_m["R2"]:.4f}   RMSE={oof_m["RMSE"]:.5f}   MAPE={oof_m["MAPE"]:.2f}%',
    fontweight='bold'
)
ax.legend(fontsize=9)
plt.colorbar(sc, ax=ax, label='Sample index')
plt.tight_layout()
plt.savefig('cv_oof_parity.png', bbox_inches='tight')
plt.show()

print(f'\nOOF  R²   = {oof_m["R2"]:.4f}')
print(f'OOF  RMSE = {oof_m["RMSE"]:.5f}')
print(f'OOF  MAPE = {oof_m["MAPE"]:.2f}%')

In [ ]:
# ── Per-fold metric bar chart ─────────────────────────────────────────────────
fold_labels = [f'Fold {i}' for i in range(1, N_FOLDS+1)]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

r2s   = [m['R2']   for m in fold_metrics]
rmses = [m['RMSE'] for m in fold_metrics]
mapes = [m['MAPE'] for m in fold_metrics]

bar_kw = dict(edgecolor='white', width=0.5)

ax1.bar(fold_labels, r2s,   color='steelblue', **bar_kw)
ax1.axhline(np.mean(r2s),   color='red', lw=2, ls='--', label=f'Mean={np.mean(r2s):.4f}')
ax1.set_ylabel('R²'); ax1.set_title('R² per Fold', fontweight='bold')
ax1.set_ylim([0, 1.05]); ax1.legend(fontsize=8)

ax2.bar(fold_labels, rmses, color='darkorange', **bar_kw)
ax2.axhline(np.mean(rmses), color='red', lw=2, ls='--', label=f'Mean={np.mean(rmses):.5f}')
ax2.set_ylabel('RMSE'); ax2.set_title('RMSE per Fold', fontweight='bold')
ax2.legend(fontsize=8)

ax3.bar(fold_labels, mapes, color='seagreen', **bar_kw)
ax3.axhline(np.mean(mapes), color='red', lw=2, ls='--', label=f'Mean={np.mean(mapes):.2f}%')
ax3.set_ylabel('MAPE (%)'); ax3.set_title('MAPE per Fold', fontweight='bold')
ax3.legend(fontsize=8)

fig.suptitle(f'{N_FOLDS}-Fold Cross-Validation Metrics', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('cv_metrics_bars.png', bbox_inches='tight')
plt.show()

## 8. Retrain on Full Dataset

In [ ]:
# ── Fit the global scaler on all training data ────────────────────────────────
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_all_s = scaler_X.fit_transform(X_all)
y_all_s = scaler_y.fit_transform(y_all)

# ── Rebuild and retrain ───────────────────────────────────────────────────────
tf.random.set_seed(42)
final_model = build_nn(X_all_s.shape[1])

# Use 10 % of training data as internal val for early stopping
split_idx = int(0.9 * len(X_all_s))
shuffle_idx = np.random.permutation(len(X_all_s))
tr_idx = shuffle_idx[:split_idx]
hv_idx = shuffle_idx[split_idx:]

final_hist = final_model.fit(
    X_all_s[tr_idx], y_all_s[tr_idx],
    validation_data=(X_all_s[hv_idx], y_all_s[hv_idx]),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1
)

best_ep = np.argmin(final_hist.history['val_loss']) + 1
print(f'\nFinal model trained — best epoch: {best_ep}')

In [ ]:
# ── Final model training curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(final_hist.history['loss'],     color='steelblue', lw=2, label='Train')
ax.semilogy(final_hist.history['val_loss'], color='tomato',    lw=2, label='Val (10 %)', alpha=0.85)
ax.axvline(best_ep - 1, color='gray', lw=1.5, ls='--', label=f'Best epoch {best_ep}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (log scale)')
ax.set_title('Final Model — Training Curve (full dataset)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('final_training_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Training-set performance ──────────────────────────────────────────────────
y_train_pred_s = final_model.predict(X_all_s, verbose=0)
y_train_pred   = scaler_y.inverse_transform(y_train_pred_s)
train_m = compute_metrics(y_all, y_train_pred)

print('Training-set metrics (full dataset):')
for k, v in train_m.items():
    print(f'  {k:>6}: {v:.4f}')

## 9. External Validation

In [ ]:
# ── Load external dataset ─────────────────────────────────────────────────────
external = read_csv_safe(EXTERNAL_FILE)
print(f'External shape   : {external.shape}')
print(f'External columns : {list(external.columns)}')
external.head()

In [ ]:
# ── Map External columns to match training feature/target names ───────────────
# Identify external target column (same logic as above)
_ext_candidates = [
    c for c in external.columns
    if any(kw in c.lower() for kw in ('recov', 'rf', 'orf', 'oil_rec', 'recovery_factor'))
]
EXT_TARGET  = _ext_candidates[0] if _ext_candidates else external.columns[-1]
EXT_FEATS   = [c for c in external.columns if c != EXT_TARGET]

print(f'External target   : {EXT_TARGET}')
print(f'External features : {EXT_FEATS}')

# Warn if column sets differ
missing_in_ext  = set(FEATURE_COLS) - set(EXT_FEATS)
extra_in_ext    = set(EXT_FEATS)    - set(FEATURE_COLS)
if missing_in_ext:
    print(f'⚠  Missing in External (will use zeros): {missing_in_ext}')
if extra_in_ext:
    print(f'⚠  Extra in External (will be ignored):  {extra_in_ext}')

In [ ]:
# ── Prepare external X, y using training feature order ───────────────────────
ext_data = external[EXT_FEATS + [EXT_TARGET]].dropna().reset_index(drop=True)

# Align external features to training feature order; fill missing with 0
ext_X_df = pd.DataFrame(index=ext_data.index)
for col in FEATURE_COLS:
    ext_X_df[col] = ext_data[col] if col in ext_data.columns else 0.0

X_ext = ext_X_df.values.astype(np.float32)
y_ext = ext_data[EXT_TARGET].values.astype(np.float32).reshape(-1, 1)

print(f'External X: {X_ext.shape}   y: {y_ext.shape}')
print(f'External y range: [{y_ext.min():.4f}, {y_ext.max():.4f}]')

In [ ]:
# ── Apply training scaler and predict ─────────────────────────────────────────
X_ext_s      = scaler_X.transform(X_ext)
y_ext_pred_s = final_model.predict(X_ext_s, verbose=0)
y_ext_pred   = scaler_y.inverse_transform(y_ext_pred_s)

ext_m = compute_metrics(y_ext, y_ext_pred)

print('='*45)
print('       EXTERNAL VALIDATION METRICS')
print('='*45)
print(f'  Samples : {len(y_ext)}')
print(f'  R²      : {ext_m["R2"]:.4f}')
print(f'  RMSE    : {ext_m["RMSE"]:.5f}')
print(f'  MAPE    : {ext_m["MAPE"]:.2f}%')
print('='*45)

## 10. Results Visualization

In [ ]:
# ── Parity plot — External validation ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_ext.ravel(), y_ext_pred.ravel(),
           color='darkorange', alpha=0.7, s=50, edgecolors='white', linewidths=0.5,
           label=f'External (n={len(y_ext)})')

lo = min(y_ext.min(), y_ext_pred.min())
hi = max(y_ext.max(), y_ext_pred.max())
pad = (hi - lo) * 0.05
ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], 'k--', lw=2, label='Ideal 1:1')

ax.set_xlim([lo-pad, hi+pad])
ax.set_ylim([lo-pad, hi+pad])
ax.set_xlabel(f'Actual {TARGET_COL}', fontsize=12)
ax.set_ylabel(f'Predicted {TARGET_COL}', fontsize=12)
ax.set_title(
    'External Validation — Parity Plot\n'
    f'R²={ext_m["R2"]:.4f}   RMSE={ext_m["RMSE"]:.5f}   MAPE={ext_m["MAPE"]:.2f}%',
    fontweight='bold'
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('ext_parity.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual analysis — External ─────────────────────────────────────────────
residuals = y_ext.ravel() - y_ext_pred.ravel()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(y_ext_pred.ravel(), residuals,
            color='steelblue', alpha=0.6, s=35, edgecolors='none')
ax1.axhline(0, color='red', lw=2, ls='--')
ax1.axhline( 2*residuals.std(), color='gray', lw=1, ls=':', label='±2σ')
ax1.axhline(-2*residuals.std(), color='gray', lw=1, ls=':')
ax1.set_xlabel('Predicted', fontsize=11)
ax1.set_ylabel('Residual (Actual − Predicted)', fontsize=11)
ax1.set_title('Residuals vs Predicted', fontweight='bold')
ax1.legend()

ax2.hist(residuals, bins=25, color='steelblue', edgecolor='white')
ax2.axvline(0,              color='red',  lw=2, ls='--', label='Zero')
ax2.axvline(residuals.mean(), color='orange', lw=2, label=f'Mean={residuals.mean():.4f}')
ax2.set_xlabel('Residual', fontsize=11)
ax2.set_ylabel('Count')
ax2.set_title('Residual Distribution', fontweight='bold')
ax2.legend()

fig.suptitle('External Validation — Residual Analysis', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('ext_residuals.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Side-by-side: CV OOF vs External parity ───────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_true, y_pred, title, color, label in [
    (ax1, y_all, oof_preds,  f'CV (OOF, {N_FOLDS}-fold)', 'steelblue', 'CV OOF'),
    (ax2, y_ext, y_ext_pred, 'External Validation',       'darkorange', 'External'),
]:
    m   = compute_metrics(y_true, y_pred)
    lo  = min(y_true.min(), y_pred.min())
    hi  = max(y_true.max(), y_pred.max())
    pad = (hi - lo) * 0.05
    ax.scatter(y_true.ravel(), y_pred.ravel(), color=color, alpha=0.65,
               s=35, edgecolors='none', label=label)
    ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], 'k--', lw=2, label='Ideal 1:1')
    ax.set_xlim([lo-pad, hi+pad]); ax.set_ylim([lo-pad, hi+pad])
    ax.set_xlabel(f'Actual {TARGET_COL}', fontsize=11)
    ax.set_ylabel(f'Predicted {TARGET_COL}', fontsize=11)
    ax.set_title(
        f'{title}\nR²={m["R2"]:.4f}   RMSE={m["RMSE"]:.4f}   MAPE={m["MAPE"]:.2f}%',
        fontweight='bold'
    )
    ax.legend(fontsize=9)

fig.suptitle('Neural Network — Cross-Validation vs External Validation', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cv_vs_ext_parity.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Comprehensive metrics comparison table ─────────────────────────────────────
summary = pd.DataFrame({
    'Dataset': [f'CV OOF ({N_FOLDS}-fold)', 'External Validation'],
    'N Samples': [len(y_all), len(y_ext)],
    'R²':   [oof_m['R2'],  ext_m['R2']],
    'RMSE': [oof_m['RMSE'], ext_m['RMSE']],
    'MAPE (%)': [oof_m['MAPE'], ext_m['MAPE']],
})
summary.set_index('Dataset', inplace=True)

print('\n=== FINAL SUMMARY ===')
print(summary.to_string())

summary.style.format({'R²': '{:.4f}', 'RMSE': '{:.5f}', 'MAPE (%)': '{:.2f}'})\
             .background_gradient(cmap='RdYlGn', subset=['R²'])\
             .background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAPE (%)'])

## 11. Save Final Model & Scalers

In [ ]:
import joblib, os

os.makedirs('nn_model', exist_ok=True)

final_model.save('nn_model/nn_oil_recovery.keras')
joblib.dump(scaler_X, 'nn_model/scaler_X.pkl')
joblib.dump(scaler_y, 'nn_model/scaler_y.pkl')

# Save OOF predictions
oof_df = data[[TARGET_COL]].copy()
oof_df['Predicted'] = oof_preds.ravel()
oof_df.to_csv('nn_model/oof_predictions.csv', index=False)

# Save external predictions
ext_df = ext_data[[EXT_TARGET]].copy()
ext_df['Predicted'] = y_ext_pred.ravel()
ext_df.to_csv('nn_model/external_predictions.csv', index=False)

print('Saved:')
print('  nn_model/nn_oil_recovery.keras')
print('  nn_model/scaler_X.pkl')
print('  nn_model/scaler_y.pkl')
print('  nn_model/oof_predictions.csv')
print('  nn_model/external_predictions.csv')